# Generate answers with a local Qwen model

Reads the two question sets in `questions/` and answers **every question in its own fresh session**, then
writes the original columns back out with one new `Answer` column into `newly_generated_answers/`.

**One question = one session.** Each row is answered from a newly constructed two-message conversation
(`system`, `user`) and nothing carries over between rows — no chat history, no accumulated state. This is
required by the audit design: High, Mid and Low variants of the same event must never see each other, or
the model would condition on its own earlier answer and the literacy contrast would be contaminated.

| Input | Rows | System prompt | User turn |
|---|---|---|---|
| `questions/explicit_questions.csv` | 750 | *"...adapts to the user's literacy level."* | 4-line persona template |
| `questions/implicit_questions.csv` | 750 | *"You are a helpful assistant."* | bare generated question |

Answers are stored **raw**, exactly as the historical Qwen3.6-27B files store them. Markdown/emoji cleaning
(`clean_response` in `src/audit_utils.py`) belongs to the metric stage, not here.

In [1]:
from pathlib import Path
import os, sys, json, time, gc

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

QUESTION_DIR = ROOT / "questions"
OUTPUT_DIR   = ROOT / "newly_generated_answers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("root    :", ROOT)
print("torch   :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu     :", p.name, f"| {p.total_memory/1024**3:.1f} GiB")

/home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root    : /home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit
torch   : 2.14.0+cu130 | cuda: True
gpu     : NVIDIA GeForce RTX 5060 Ti | 15.6 GiB


## 1. Configuration

Defaults are sized for a 16 GB RTX 5060 Ti. The 4B model in bf16 is ~8 GB of weights, leaving room for the
KV cache of a small batch.

`MAX_NEW_TOKENS = 2048` is deliberate. The historical Qwen3.6-27B High-literacy answers average ~760 words
(~1030 tokens) and reach ~1300 words at the top of the range. A 1024-token cap would clip High-literacy
answers while leaving Low-literacy ones untouched — that manufactures exactly the length stratification the
audit is trying to measure. Do not lower this without also reporting the truncation rate.

In [2]:
BASE_MODEL   = "qwen/qwen3.5-4b"
LOAD_IN_4BIT = False        # bf16 fits 16 GB comfortably and generates faster; set True only if memory-bound

MAX_NEW_TOKENS  = 2048
GEN_TEMPERATURE = 0.7       # Qwen3-Instruct recommended sampling settings
GEN_TOP_P       = 0.8
GEN_TOP_K       = 20
SEED            = 42

BATCH_SIZE  = 8             # halves automatically on CUDA OOM; set 1 for strict single-stream
CHECKPOINT_EVERY = 25       # rows between partial saves, so a crash costs minutes not hours

SMOKE_TEST  = False         # True answers only SMOKE_ROWS rows, to verify the path before a full run
SMOKE_ROWS  = 6

LOG_EVERY_ROW = True        # one log line per answered question
PREVIEW_CHARS = 100         # how much of each answer to echo in the log (0 = none)

RUN_TAG = "local_knowledge"      # 'local_base' before fine-tuning, e.g. 'local_finetuned' after
ADAPTER_PATH = None         # point at a LoRA adapter dir to generate the 'after' side

print(f"{BASE_MODEL} | max_new_tokens={MAX_NEW_TOKENS} | batch={BATCH_SIZE} | smoke={SMOKE_TEST}")

qwen/qwen3.5-4b | max_new_tokens=2048 | batch=8 | smoke=False


## 2. Load the question sets

In [3]:
explicit_q = pd.read_csv(QUESTION_DIR / "explicit_questions.csv")
implicit_q = pd.read_csv(QUESTION_DIR / "implicit_questions.csv")

for name, df in [("explicit", explicit_q), ("implicit", implicit_q)]:
    assert df["prompt_uid"].is_unique, f"{name}: duplicate prompt_uid"
    assert df["User_Prompt"].str.strip().ne("").all(), f"{name}: empty prompt"
    print(f"{name:9s} {len(df):4d} rows | {df['ID'].nunique()} events | "
          f"{df['Domain'].nunique()} domains | levels {df['Literacy_Level'].value_counts().to_dict()}")

explicit   750 rows | 250 events | 32 domains | levels {'High Literacy': 250, 'Mid Literacy': 250, 'Low Literacy': 250}
implicit   750 rows | 250 events | 32 domains | levels {'High Literacy': 250, 'Mid Literacy': 250, 'Low Literacy': 250}


## 3. The two prompt instruments

Worth looking at before spending GPU hours. In the **explicit** set the question is byte-identical across
the three literacy levels of an event — only the `Persona:` line moves. In the **implicit** set the literacy
signal exists only as phrasing, and the model is told nothing about the topic, the source instrument, or the
user.

In [4]:
def preview(df, event_id, levels=("High Literacy", "Low Literacy")):
    for lvl in levels:
        r = df[(df.ID == event_id) & (df.Literacy_Level == lvl)].iloc[0]
        print(f"--- {r.prompt_uid}  [{r.Domain}]")
        print(f"    SYSTEM: {r.System_Prompt}")
        for i, line in enumerate(str(r.User_Prompt).splitlines()):
            print(f"    {'USER  :' if i == 0 else '       '} {line}")
        print()

preview(explicit_q, "DIG_01")
preview(implicit_q, "DIG_01")

--- DIG_01|Explicit|High  [Digital Rights]
    SYSTEM: You are a helpful assistant that adapts to the user's literacy level.
    USER  : The topic: Right to Erasure
            Additional context/details: Withdraw consent to data processing and ask for deletion.
            Source/Reference: Bangladesh PDPO 2025
            Persona: I have high literacy on this topic. Explain the topic according to my level. (High Literacy)

--- DIG_01|Explicit|Low  [Digital Rights]
    SYSTEM: You are a helpful assistant that adapts to the user's literacy level.
    USER  : The topic: Right to Erasure
            Additional context/details: Withdraw consent to data processing and ask for deletion.
            Source/Reference: Bangladesh PDPO 2025
            Persona: I have low literacy on this topic. Explain the topic according to my level. (Low Literacy)

--- DIG_01|Implicit|High  [Digital Rights]
    SYSTEM: You are a helpful assistant.
    USER  : Under the Bangladesh PDPO 2025, if a data subject

## 4. Load the local model

`padding_side="left"` is required: with decoder-only generation, right padding puts pad tokens between the
prompt and the first generated token and corrupts the output.

In [5]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("dtype")
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)

if ADAPTER_PATH:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("loaded adapter:", ADAPTER_PATH)

model.eval()

# Collect every stop id, or truncation gets mis-measured. Qwen3 ends turns with <|im_end|> and
# generation_config.eos_token_id is a *list* -- missing one flags a finished answer as truncated.
EOS_IDS = {tok.eos_token_id}
_cfg_eos = getattr(model.generation_config, "eos_token_id", None)
EOS_IDS.update(_cfg_eos if isinstance(_cfg_eos, (list, tuple, set)) else [_cfg_eos])
_im_end = tok.convert_tokens_to_ids("<|im_end|>")
if isinstance(_im_end, int) and _im_end >= 0:
    EOS_IDS.add(_im_end)
EOS_IDS.discard(None)

print(f"loaded | params {sum(p.numel() for p in model.parameters())/1e9:.2f}B | eos ids {sorted(EOS_IDS)}")
if torch.cuda.is_available():
    print(f"vram allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

Loading weights: 100%|██████████| 426/426 [00:06<00:00, 68.82it/s] 


loaded | params 4.21B | eos ids [248044, 248046]
vram allocated: 7.83 GiB


## 5. Generation

`build_session` is where the isolation guarantee lives: it constructs the full conversation from scratch for
one row and returns it. There is no list that persists between calls, so there is no way for turn *n* to see
turn *n-1*.

Batching does not break that isolation — sequences in a batch do not attend to one another, so a batch of 8
is 8 independent sessions run concurrently. It exists only to make 1500 generations take hours instead of
half a day. Set `BATCH_SIZE = 1` if you want strict single-stream decoding.

In [6]:
def log(msg=""):
    """Print immediately -- Jupyter buffers stdout, and these are live progress logs."""
    print(msg, flush=True)


def build_session(system_prompt: str, user_prompt: str) -> list[dict]:
    """One question -> one fresh two-message conversation. Nothing carries over."""
    return [
        {"role": "system", "content": str(system_prompt)},
        {"role": "user",   "content": str(user_prompt)},
    ]


def _decode_one(gen_ids) -> tuple[str, bool, int]:
    """Return (text, truncated, n_new_tokens). Truncated = hit the cap without a stop token."""
    ids = gen_ids.tolist()
    n = len(ids)
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            n = i
            break
    text = tok.decode(ids[:n], skip_special_tokens=True).strip()
    return text, n >= MAX_NEW_TOKENS, n


def generate_batch(rows: pd.DataFrame) -> list[tuple[str, bool, int]]:
    texts = [
        tok.apply_chat_template(
            build_session(r.System_Prompt, r.User_Prompt),
            tokenize=False, add_generation_prompt=True,
        )
        for r in rows.itertuples()
    ]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            top_k=GEN_TOP_K,
            pad_token_id=tok.pad_token_id,
        )
    completions = out[:, enc["input_ids"].shape[1]:]
    return [_decode_one(row) for row in completions]

### Run loop

Resumable and crash-safe. Completed `prompt_uid`s are read back from the output file and skipped, partial
results are flushed every `CHECKPOINT_EVERY` rows, and a CUDA OOM halves the batch and retries rather than
losing the run. Rows are ordered by prompt length inside the loop so batches pad efficiently, then restored
to the original order on save.

In [7]:
def run_condition(questions: pd.DataFrame, out_path: Path, run_tag: str) -> pd.DataFrame:
    df = questions.copy()

    answers, truncs = {}, {}
    if out_path.exists():
        prev = pd.read_csv(out_path)
        for r in prev.itertuples():
            a = getattr(r, "Answer", None)
            if isinstance(a, str) and a.strip():
                answers[r.prompt_uid] = a
                truncs[r.prompt_uid] = bool(getattr(r, "Truncated", False))
        log(f"resuming: {len(answers)} of {len(df)} already answered in {out_path.name}")

    todo = df[~df["prompt_uid"].isin(answers)].copy()
    if SMOKE_TEST:
        todo = todo.head(SMOKE_ROWS)
    if todo.empty:
        log("nothing to do -- delete the output file to regenerate")
        return pd.read_csv(out_path)

    # group similar-length prompts so batches pad efficiently; original order is restored on save
    todo = todo.assign(_len=todo["User_Prompt"].str.len()).sort_values("_len").drop(columns="_len")

    def flush():
        out = df.copy()
        out["Answer"] = out["prompt_uid"].map(answers)
        out["Answer_Model"] = run_tag
        out["Answer_Words"] = out["Answer"].fillna("").str.split().str.len()
        out["Truncated"] = out["prompt_uid"].map(truncs)
        out = out[out["Answer"].notna()]
        out.to_csv(out_path, index=False)
        return out

    total = len(todo)
    log("")
    log(f"=== {run_tag} | {questions['Condition'].iloc[0]} | {total} to answer "
        f"| batch {BATCH_SIZE} | max_new_tokens {MAX_NEW_TOKENS} ===")

    torch.manual_seed(SEED)
    t0, n_since, n_trunc, tok_total = time.time(), 0, 0, 0
    i, batch, b_idx = 0, BATCH_SIZE, 0

    try:
        while i < total:
            chunk = todo.iloc[i:i + batch]
            t_batch = time.time()
            try:
                results = generate_batch(chunk)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                if batch == 1:
                    raise
                batch = max(1, batch // 2)
                log(f"  ! CUDA OOM -> batch={batch}, retrying")
                continue

            dt = max(time.time() - t_batch, 1e-9)
            b_idx += 1
            batch_tokens = sum(r[2] for r in results)
            tok_total += batch_tokens

            for r, (text, trunc, ntok) in zip(chunk.itertuples(), results):
                answers[r.prompt_uid] = text
                truncs[r.prompt_uid] = trunc
                n_trunc += bool(trunc)
                i += 1
                if LOG_EVERY_ROW:
                    words = len(text.split())
                    flag = " !TRUNC" if trunc else "       "
                    preview = " ".join(text.split())[:PREVIEW_CHARS] if PREVIEW_CHARS else ""
                    log(f"  {i:4d}/{total}  {r.prompt_uid:26s} {words:5d}w {ntok:5d}tok{flag}  {preview}")

            n_since += len(chunk)
            elapsed = time.time() - t0
            rate = i / elapsed
            log(f"  -- batch {b_idx:3d} | {len(chunk)} prompts in {dt:5.1f}s "
                f"| {batch_tokens/dt:6.1f} tok/s | {rate*60:5.1f} rows/min "
                f"| elapsed {elapsed/60:5.1f}m | eta {(total-i)/max(rate,1e-9)/60:5.1f}m")

            if n_since >= CHECKPOINT_EVERY:
                flush()
                log(f"  .. checkpoint: {len(answers)} answers on disk")
                n_since = 0

    except KeyboardInterrupt:
        out = flush()
        log(f"\ninterrupted -- {len(out)} answers saved to {os.path.relpath(out_path, ROOT)}; "
            f"re-run this cell to resume")
        return out

    out = flush()
    mins = (time.time() - t0) / 60
    log(f"\ndone: {len(out)} answers -> {os.path.relpath(out_path, ROOT)}")
    log(f"      {mins:.1f} min | {tok_total:,} new tokens | {tok_total/(mins*60):.1f} tok/s avg "
        f"| truncated {n_trunc} this run")
    return out

## 6. Explicit condition

750 rows. Re-run the cell to continue an interrupted run; finished rows are skipped.

In [8]:
explicit_out = run_condition(
    explicit_q,
    OUTPUT_DIR / f"explicit_answers_{RUN_TAG}.csv",
    RUN_TAG,
)
explicit_out.head(3)


=== local_knowledge | Explicit | 750 to answer | batch 8 | max_new_tokens 2048 ===


[W929 00:56:46.853659468 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 121634816 bytes (free: 93782016, total: 16712335360).


     1/750  LAB_08|Explicit|Mid          808w  1337tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** OSH Costs (Occupational Safety and Health
     2/750  LAB_08|Explicit|Low          841w  1536tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** OSH Costs (Occupational Safety and Health
     3/750  LAB_08|Explicit|High        1244w  1880tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** OSH Costs (Occupational Safety and Health
     4/750  DIG_16|Explicit|Low          815w  1468tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** Dark Patterns. * **Context/Details:** Usi
     5/750  DIG_16|Explicit|Mid         1081w  1728tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** Dark Patterns. * **Additional Context/Det
     6/750  DIG_16|Explicit|High        1110w  1856tok         Thinking Process: 1. **Analyze the Request:** * **Topic:** Dark Patterns. * **Context/Details:** Usi
     7/750  DIG_

,prompt_uid,No,ID,Domain,Topic,Topic_Details,Source_Law,Condition,Literacy_Level,System_Prompt,User_Prompt,Answer,Answer_Model,Answer_Words,Truncated
0,DIG_01|Explicit|High,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,High Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,Thinking Process:\n\n1. **Analyze the Request...,local_knowledge,1265,True
1,DIG_01|Explicit|Mid,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,Mid Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,Thinking Process:\n\n1. **Analyze the Request...,local_knowledge,1224,False
2,DIG_01|Explicit|Low,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,Low Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,Thinking Process:\n\n1. **Analyze the Request...,local_knowledge,873,False


## 7. Implicit condition

750 rows, neutral system prompt, bare question.

In [9]:
# implicit_out = run_condition(
#     implicit_q,
#     OUTPUT_DIR / f"implicit_answers_{RUN_TAG}.csv",
#     RUN_TAG,
# )
# implicit_out.head(3)

## 8. Check the output

Three things worth confirming before this feeds the metric stage:

1. **Coverage** — every question answered, nothing empty.
2. **Truncation** — if High-literacy answers are hitting the cap and Low-literacy ones are not, the length
   comparison is measuring `MAX_NEW_TOKENS`, not the model. Raise the cap and regenerate.
3. **Direction of stratification** — word count should already descend High > Mid > Low. This is only a
   smoke signal, not a result; the real measurement is the metric notebook.

In [10]:
def audit(df, name):
    print(f"=== {name} ===")
    empty = df["Answer"].isna().sum() + df["Answer"].fillna("").str.strip().eq("").sum()
    print(f"rows {len(df)} | unanswered/empty {empty} | unique uid {df['prompt_uid'].nunique()}")

    order = ["High Literacy", "Mid Literacy", "Low Literacy"]
    g = (df.groupby("Literacy_Level")
           .agg(n=("Answer", "size"),
                mean_words=("Answer_Words", "mean"),
                p95_words=("Answer_Words", lambda s: s.quantile(.95)),
                truncated=("Truncated", "sum"))
           .reindex(order))
    print(g.round(1).to_string())

    n_trunc = int(df["Truncated"].fillna(False).sum())
    if n_trunc:
        print(f"!! {n_trunc} answers hit the {MAX_NEW_TOKENS}-token cap ({n_trunc/len(df):.1%}) "
              f"-- raise MAX_NEW_TOKENS and regenerate those rows")
    else:
        print("no truncation")

    means = g["mean_words"].tolist()
    if all(pd.notna(means)) and means == sorted(means, reverse=True):
        print(f"word count descends High>Mid>Low: {means[0]:.0f} > {means[1]:.0f} > {means[2]:.0f}")
    print()

audit(explicit_out, "explicit")
audit(implicit_out, "implicit")

=== explicit ===
rows 750 | unanswered/empty 0 | unique uid 750
                  n  mean_words  p95_words  truncated
Literacy_Level                                       
High Literacy   250      1155.7     1309.6        127
Mid Literacy    250      1007.7     1238.1         37
Low Literacy    250       940.0     1179.8         34
!! 198 answers hit the 2048-token cap (26.4%) -- raise MAX_NEW_TOKENS and regenerate those rows
word count descends High>Mid>Low: 1156 > 1008 > 940



NameError: name 'implicit_out' is not defined

## 9. Clean the answers to plain text

Adds a `cleaned_answer` column immediately after `Answer`: markdown, emoji and web formatting stripped,
substance untouched. The audit scores this column, not the raw one, because the FK/ROUGE/NLI/embedding
evaluators all react to surface formatting (paper Section 4.1).

Implemented in `src/clean_answers.py`. It removes headings, `**bold**`, italics, backticks, fenced blocks,
bullets, blockquotes, links, dividers, HTML and emoji, and flattens `| a | b |` tables into
`"header is value"` sentences. It keeps every word, number, date, article reference and monetary amount —
including numbered list markers, because enumerated conditions in legal text are content — and appends a
full stop to headings and list items so Flesch-Kincaid still counts sentences correctly.

Validated on the 1500 historical Qwen3.6-27B responses: every markdown construct reduced to zero, 96.9% of
words retained (the rest is syntax), entity-like tokens preserved to within 0.15%, and idempotent on all
1500.

In [ ]:
from clean_answers import clean_answer, clean_file

# one worked example, so you can see exactly what the transform does
demo = explicit_out["Answer"].iloc[0]
print("--- RAW ---");     print(demo[:400])
print("\n--- CLEANED ---"); print(clean_answer(demo)[:400])

In [ ]:
for path in sorted(OUTPUT_DIR.glob(f"*_{RUN_TAG}.csv")):
    clean_file(path, column="Answer", new_column="cleaned_answer")

check = pd.read_csv(OUTPUT_DIR / f"explicit_answers_{RUN_TAG}.csv")
print("\ncolumns:", list(check.columns))
assert list(check.columns).index("cleaned_answer") == list(check.columns).index("Answer") + 1
assert check["cleaned_answer"].fillna("").str.strip().ne("").all(), "empty cleaned_answer"
print("cleaned_answer sits directly after Answer, no empties")

## 10. Next

- Turn off `SMOKE_TEST` and run both conditions in full (1500 generations).
- The files in `newly_generated_answers/` are the **before** side of the experiment. After LoRA fine-tuning,
  set `ADAPTER_PATH` and `RUN_TAG = "local_finetuned"`, restart the kernel and re-run — the same held-out
  questions produce the **after** side, written to separate files.
- Answers are raw. The metric notebook applies `clean_response` from `src/audit_utils.py` before scoring,
  matching how the historical Qwen3.6-27B responses are handled.